# Pre- vs. post-SHINE cohort comparison: semantic and sensory decomposition

Does SHINE change *what perceived similarity is made of*? Each cohort's behavioral RDM is regressed on semantic and sensory predictor RDMs (pre-registered H3, following Shoham et al. 2024), and the weights are compared between the cohorts, for **retained subjects only** (production cohort, `status == "full data"`, `task_version >= 4.0`).<br>
Companion notebooks:
- `0_sensory.ipynb` - does SHINE change the stimuli themselves?
- `1_demographics_and_behavior.ipynb` - do they behave the same?
- `2_data_quality.ipynb` - is the data equally good?
- `3_geometry.ipynb` - do they give the same similarity structure?
- `4_semantic_hierarchy.ipynb` - are they equally similar to the Kiani hierarchy?
- `5_ground_truth.ipynb` - are they equally similar to earlier ratings and recordings?

Backend helpers live in `decomposition.py`.

**Model.** `perceptual ~ 1 + semantic + [CLIP] + pixels`, fit per cohort by **non-negative least squares** on the condensed RDMs (the pre-registered estimator: a weight is a contribution to perceived distance and cannot be negative, and the constraint returns exact zeros for predictors that add nothing). The perceptual RDM and every predictor are z-scored over the pairs of the fit, so coefficients are standardized betas, comparable across predictors and cohorts, and the intercept is 0. Pixels and CLIP are each cohort's own SHINE variant; the semantic RDMs are shared. The **baseline** semantic predictor is the Kiani-Mur tree distance (`km`); WordNet (`wn`) and both together (`km+wn`) are robustness checks at the bottom. Every specification is run without and with CLIP.

**Inference.** Participant bootstrap (each cohort resampled on its own, the combined RDM rebuilt, every model refit), percentile 95% CIs. The pre minus post difference of each weight gets a 95% CI (H3(i) asks whether the pixel weight is *higher* pre-SHINE) and a TOST equivalence test with the project's 80%-power bound (H3(ii) asks whether the semantic weight is *unchanged*). **Primary pairs: each cohort's own observed pairs**; the pairs both cohorts observed are a sensitivity check (section 7).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s1_pre_vs_post_shine/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from analysis.utils.stats import spearman_rho
from analysis.s0_build_rdms.behavior.spam_ratings import build_behavioral_rdm
from analysis.s1_pre_vs_post_shine import decomposition as dec
from analysis.s1_pre_vs_post_shine.demographics_and_behavior import COHORTS, COHORT_LABEL, load_retained_cohorts
from analysis.s1_pre_vs_post_shine.geometry import parse_subject_trials
from analysis.s1_pre_vs_post_shine.reference_agreement import REFERENCE_CACHE_DIR, split_by_cohort

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

## 0. Load data and fit

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in COHORTS}
predictors = {c: dec.predictors_for(c) for c in COHORTS}

cohorts = load_retained_cohorts(cache_dir)
subject_ids = {c: cohorts["participant_ids"][c].tolist() for c in COHORTS}
trials_by_subject, n_conds = parse_subject_trials(cohorts["df_trials"], subject_ids)
trials = split_by_cohort(trials_by_subject, subject_ids)

SEED, N_BOOT, N_JOBS = 0, 1000, 6
cache = lambda name: REFERENCE_CACHE_DIR / name
joint = ~np.isnan(rdm["pre"]) & ~np.isnan(rdm["post"])
masks = {"own": None, "joint": joint}
n_pre, n_post = len(subject_ids["pre"]), len(subject_ids["post"])
print(f"pairs observed: pre {int((~np.isnan(rdm['pre'])).sum()):,}, post {int((~np.isnan(rdm['post'])).sum()):,}, both {int(joint.sum()):,}")

[behavioral] variant='pre_shine': 79 participants.
[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).
[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)
[behavioral] variant='post_shine': 76 participants.
[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).
[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 26

In [3]:
observed = dec.observed_decomposition(rdm["pre"], rdm["post"], predictors["pre"], predictors["post"], pair_masks=masks)
boots = dec.bootstrap_decomposition(
    trials["pre"], trials["post"], predictors["pre"], predictors["post"], n_conds,
    n_boot=N_BOOT, seed=SEED, pair_masks=masks, n_jobs=N_JOBS, cache_path=cache(f"decomposition_bootstrap_n{N_BOOT}_seed{SEED}.pkl"),
)
stats = ("beta", "beta_ols", "partial_r", "delta_r2", "r2")
estimates = {s: dec.cohort_estimates(observed, boots, stat=s) for s in stats}
differences = {s: dec.difference_table(observed, boots, n_pre, n_post, stat=s) for s in stats}
own_betas = estimates["beta"][estimates["beta"]["pairs"] == "own"]
print(f"{N_BOOT} bootstrap draws, {len(boots):,} rows")

1000 bootstrap draws, 34,000 rows


## 1. How collinear are the predictors?

Pairwise Spearman rho between the predictor RDMs of each cohort's own variant. The pre-registration expected them to be close to orthogonal; the stronger the correlation, the less the individual weights can be trusted apart (the shared variance is split arbitrarily, and NNLS may zero one of two redundant predictors).

In [4]:
keys = list(dec.TERMS)
for c in COHORTS:
    print(COHORT_LABEL[c])
    print(pd.DataFrame([[spearman_rho(predictors[c][a], predictors[c][b]) for b in keys] for a in keys],
                       index=[dec.TERM_LABEL[k] for k in keys], columns=[dec.TERM_LABEL[k] for k in keys]).round(2).to_string(), "\n")

Pre-SHINE
                    semantic (KM)  semantic (WordNet)  CLIP  pixels
semantic (KM)                1.00                0.29  0.35    0.09
semantic (WordNet)           0.29                1.00  0.05   -0.07
CLIP                         0.35                0.05  1.00    0.24
pixels                       0.09               -0.07  0.24    1.00 

Post-SHINE
                    semantic (KM)  semantic (WordNet)  CLIP  pixels
semantic (KM)                1.00                0.29  0.31    0.05
semantic (WordNet)           0.29                1.00  0.04    0.00
CLIP                         0.31                0.04  1.00    0.18
pixels                       0.05                0.00  0.18    1.00 



## 2. All weights at a glance

Standardized NNLS betas of the six models (each cohort's own pairs, 95% participant-bootstrap CIs). The panels share one x axis; the tables below give the numbers and the pre - post comparisons.

In [5]:
dec.fig_estimates(own_betas, list(dec.MODELS), x_title="standardized beta (NNLS)").show()

## 3. Baseline: perceptual ~ semantic (KM) + pixels

Standardized NNLS betas per cohort (95% participant-bootstrap CI), their pre minus post difference (95% CI), whether that CI lies entirely above 0, and whether the difference is equivalent to 0 within the 80%-power bound.

In [6]:
r2 = estimates["r2"][estimates["r2"]["pairs"] == "own"].drop_duplicates(["model", "cohort"])
print("R^2 (own pairs):")
print(r2[r2["model"] == "km"].pivot(index="model", columns="cohort", values="estimate").round(3).to_string(), "\n")
print(dec.model_summary(estimates["beta"], differences["beta"], "km").to_string())

R^2 (own pairs):
cohort   post    pre
model               
km      0.135  0.125 

                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.342 [0.300, 0.342]  0.347 [0.301, 0.351]  -0.006 [-0.038, +0.029]       False        True   0.05
pixels         0.061 [0.050, 0.065]  0.096 [0.080, 0.099]  -0.035 [-0.044, -0.020]       False       False   0.02


## 4. Adding CLIP: perceptual ~ semantic (KM) + CLIP + pixels (confirmatory H3)

- **H3(i)**: the pixel weight is higher pre-SHINE, i.e. the 95% CI of (pre - post) lies above 0.
- **H3(ii)**: the semantic weight is unchanged, i.e. the pre - post difference is equivalent to 0.
- **H3(iii)**: the CLIP weight, reported without a directional claim.

In [7]:
print("R^2 (own pairs):")
print(r2[r2["model"] == "km+clip"].pivot(index="model", columns="cohort", values="estimate").round(3).to_string(), "\n")
print(dec.model_summary(estimates["beta"], differences["beta"], "km+clip").to_string())

R^2 (own pairs):
cohort    post    pre
model                
km+clip  0.163  0.151 

                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.272 [0.234, 0.275]  0.280 [0.237, 0.287]  -0.008 [-0.039, +0.028]       False        True   0.05
CLIP           0.180 [0.157, 0.180]  0.186 [0.165, 0.188]  -0.006 [-0.024, +0.008]       False        True   0.03
pixels         0.024 [0.016, 0.030]  0.065 [0.051, 0.069]  -0.041 [-0.049, -0.026]       False       False   0.02


Pre minus post weight of every predictor in the three CLIP models (the confirmatory `km+clip`, and its WordNet and KM + WordNet robustness versions): the 90% TOST interval, with the equivalence bound as two ticks on the same row. Blue rows are equivalent to 0 within the bound, red rows are not.

In [8]:
dec.fig_difference_forest(
    differences["beta"].query("pairs == 'own'"), ["km+clip", "wn+clip", "km+wn+clip"], x_title="pre - post, standardized beta",
).show()

## 5. Variance attribution

Pre-registered entry order: pixels, then CLIP, then the semantic predictor; **delta R^2** is the extra variance each step explains (with two semantic predictors KM enters before WordNet, so WordNet's share is what it adds beyond KM). **Partial correlations** are each predictor's unique association with the perceptual RDM given the others. Both come from the same bootstrap draws (CIs are 95% percentile intervals).

In [9]:
dec.fig_delta_r2(observed[observed["pairs"] == "own"]).show()
print("delta R^2, KM + CLIP + pixels:")
print(dec.model_summary(estimates["delta_r2"], differences["delta_r2"], "km+clip").to_string())
print("\npartial correlation, KM + CLIP + pixels:")
print(dec.model_summary(estimates["partial_r"], differences["partial_r"], "km+clip").to_string())

delta R^2, KM + CLIP + pixels:
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.061 [0.046, 0.063]  0.067 [0.048, 0.071]  -0.006 [-0.019, +0.010]       False        True   0.03
CLIP           0.079 [0.063, 0.077]  0.082 [0.065, 0.080]  -0.002 [-0.013, +0.008]       False        True   0.02
pixels         0.010 [0.008, 0.010]  0.015 [0.011, 0.015]  -0.005 [-0.007, -0.002]       False        True   0.01

partial correlation, KM + CLIP + pixels:
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.260 [0.222, 0.262]  0.273 [0.229, 0.278]  -0.013 [-0.044, +0.022]       False        True   0.05
CLIP           

## 6. Robustness: WordNet as the semantic predictor, and both semantic predictors

WordNet shortest-path distance (`wn`) instead of KM, then both in one model. `rdm_vs_hierarchy` found KM and WordNet to carry largely complementary information (rho(KM, WN) is about 0.3), so the combined fit need not zero either of them: the NNLS result is reported as it comes out, together with how often each weight is exactly 0 across the bootstrap draws.

In [10]:
for model in ("wn", "wn+clip"):
    print(model)
    print(dec.model_summary(estimates["beta"], differences["beta"], model).to_string(), "\n")

wn
                                     pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                                  
semantic (WordNet)  0.218 [0.190, 0.219]  0.197 [0.169, 0.201]  +0.021 [-0.001, +0.041]       False        True   0.04
pixels              0.106 [0.092, 0.107]  0.110 [0.094, 0.112]  -0.005 [-0.016, +0.008]       False        True   0.02 

wn+clip
                                     pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                                  
semantic (WordNet)  0.173 [0.149, 0.176]  0.160 [0.135, 0.165]  +0.013 [-0.008, +0.034]       False        True   0.04
CLIP                0.259 [0.229, 0.256]  0.268 [0.238, 0.267]  -0.009 [-0.030, +0.010]       False        True   0.03
pixels              0.040 [0.031, 0

In [11]:
for model in ("km+wn", "km+wn+clip"):
    print(model)
    print(dec.model_summary(estimates["beta"], differences["beta"], model).to_string(), "\n")
zeros = dec.zero_fraction(boots).query("pairs == 'own' and model in ['km+wn', 'km+wn+clip']")
zeros["term"] = zeros["term"].map(dec.TERM_LABEL)
print("share of bootstrap draws in which the beta is exactly 0 (pre = a, post = b):")
print(zeros.drop(columns="pairs").rename(columns={"a": "pre", "b": "post"}).round(3).to_string(index=False))

km+wn
                                     pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                                  
semantic (KM)       0.302 [0.262, 0.304]  0.317 [0.271, 0.322]  -0.015 [-0.046, +0.021]       False        True   0.05
semantic (WordNet)  0.116 [0.096, 0.121]  0.092 [0.070, 0.101]  +0.024 [+0.002, +0.042]        True       False   0.03
pixels              0.068 [0.057, 0.072]  0.093 [0.077, 0.096]  -0.025 [-0.034, -0.010]       False       False   0.02 

km+wn+clip
                                     pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                                  
semantic (KM)       0.237 [0.203, 0.243]  0.251 [0.208, 0.260]  -0.014 [-0.044, +0.023]       False        True   0.05
semantic (WordNet)  0.108 [0.

## 7. Sensitivity: only the pairs both cohorts observed

The cohorts observed different pairs (see `3_geometry.ipynb`). Restricting both fits to the pairs both observed removes that difference; compare with sections 3 and 4.

In [12]:
for model in ("km", "km+clip"):
    print(model, "(jointly observed pairs)")
    print(dec.model_summary(estimates["beta"], differences["beta"], model, pairs="joint").to_string(), "\n")

km (jointly observed pairs)
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.343 [0.302, 0.344]  0.344 [0.297, 0.349]  -0.001 [-0.033, +0.033]       False        True   0.06
pixels         0.060 [0.049, 0.065]  0.093 [0.077, 0.097]  -0.033 [-0.043, -0.018]       False       False   0.02 

km+clip (jointly observed pairs)
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.273 [0.236, 0.277]  0.277 [0.233, 0.285]  -0.004 [-0.036, +0.031]       False        True   0.05
CLIP           0.182 [0.159, 0.180]  0.183 [0.162, 0.186]  -0.001 [-0.021, +0.011]       False        True   0.03
pixels         0.022 [0.0

## 8. Sensitivity: unconstrained OLS instead of NNLS

The same standardized fits without the non-negativity constraint (same pairs, same bootstrap draws). Where the OLS and NNLS betas agree the constraint is inactive, so the pre-registered estimator is not driving the result; a negative OLS beta marks a predictor NNLS holds at 0.

In [13]:
fits = observed[observed["pairs"] == "own"]
gap = (fits["beta_a"] - fits["beta_ols_a"]).abs().groupby(fits["model"], sort=False).max()
gap_b = (fits["beta_b"] - fits["beta_ols_b"]).abs().groupby(fits["model"], sort=False).max()
print("largest |NNLS beta - OLS beta| per model:")
print(pd.DataFrame({"pre": gap, "post": gap_b}).round(4).to_string(), "\n")
for model in ("km", "km+clip", "km+wn+clip"):
    print(model, "(OLS)")
    print(dec.model_summary(estimates["beta_ols"], differences["beta_ols"], model).to_string(), "\n")

largest |NNLS beta - OLS beta| per model:
            pre  post
model                
km          0.0   0.0
km+clip     0.0   0.0
wn          0.0   0.0
wn+clip     0.0   0.0
km+wn       0.0   0.0
km+wn+clip  0.0   0.0 

km (OLS)
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.342 [0.300, 0.342]  0.347 [0.301, 0.351]  -0.006 [-0.038, +0.029]       False        True   0.05
pixels         0.061 [0.050, 0.065]  0.096 [0.080, 0.099]  -0.035 [-0.044, -0.020]       False       False   0.02 

km+clip (OLS)
                                pre                  post               pre - post  pre > post  equivalent  bound
term                                                                                                             
semantic (KM)  0.272 [0.234, 0.275]  0.280 [0.237, 0.287]  -0.008 [-0.0

## 9. Takeaways

Standardized weights, each cohort's own pairs, `km+clip` unless stated (pre - post with 95% CI; equivalence bound in brackets).

- **H3(i) is not supported, and the direction is reversed.** The pixel weight is *higher post-SHINE* (0.065 vs 0.024, pre - post = -0.041 [-0.049, -0.026]), the opposite of the pre-registered expectation. The same holds in all six models and on the jointly observed pairs, and the difference is not equivalent to 0 (bound 0.02). The pixel weight is small in both cohorts, so this is a small absolute shift.
- **H3(ii) is supported.** The semantic (KM) weight is equivalent across cohorts (-0.008 [-0.039, +0.028], bound 0.05), as is the CLIP weight (H3(iii), -0.006 [-0.024, +0.008], bound 0.03).
- **Variance attribution.** Unique contributions (delta R^2) are about 0.06 to 0.07 for semantic (KM), 0.08 for CLIP and 0.01 to 0.015 for pixels, with total R^2 of 0.151 (pre) and 0.163 (post). Pixels add slightly more post-SHINE (-0.005 [-0.007, -0.002]); semantic and CLIP shares are equivalent.
- **Robustness.** WordNet alone reproduces the pattern (semantic weight equivalent, pixels higher post when CLIP is in the model). With KM and WordNet together, **neither beta is 0 in any bootstrap draw** (KM about 0.30, WordNet about 0.11): the two carry complementary information (rho about 0.29), so the expected "one of them drops out" does not occur. WordNet's weight is nominally higher pre-SHINE (+0.020 [-0.003, +0.038] with CLIP); this is secondary and its equivalence test is inconclusive.
- **Estimator.** NNLS and OLS betas are identical to 4 decimals in every model (no constraint is active), so the pre-registered estimator does not drive any result.
- **Caveat.** Percentile CIs of the single-cohort weights are not centered on the estimates (resampling participants lowers the number of distinct pairs), so lean on the pre - post differences, which are far less affected.